# 📚 GUÍA COMPLETA: ORDEN CORRECTO DE ANÁLISIS DE DATOS

## 🎯 FLUJO GENERAL DESDE CERO HASTA MODELADO

## 📋 SECUENCIA COMPLETA PASO A PASO

```
1. LIMPIEZA DE DATOS (antes del split)
   - Eliminar duplicados
   - Convertir tipos de datos
   - Codificar variables categóricas
   - Reemplazar valores imposibles (ej: 0 en glucosa)
   ↓
   
2. TRAIN/TEST SPLIT ⭐ (MOMENTO CRÍTICO)
   - Dividir antes de cualquier transformación estadística
   - Evitar data leakage
   ↓
   
3. IMPUTACIÓN (solo fit en train)
   - Entrenar imputer solo con X_train
   - Aplicar transform a X_test
   ↓
   
4. CORRELACIONES (solo en train) 🔴 PRIMERO
   - Calcular matriz de correlación
   - Eliminar variables con |r| > 0.8
   ↓
   
5. VIF (solo en train)
   - Calcular Variance Inflation Factor
   - Eliminar variables con VIF > 10
   ↓
   
6. MÉTODOS DE SELECCIÓN (solo en train) 🔵 DESPUÉS
   - Random Forest Feature Importance
   - Permutation Importance
   - SHAP values
   ↓
   
7. ESCALADO (fit en train, transform en test)
   - StandardScaler o RobustScaler
   ↓
   
8. MODELADO
   - Entrenar modelo final
```

---

# 🔴 PASO 4: CORRELACIONES (PRIMERO)

## ¿Por qué primero?
- ✅ Detecta **multicolinealidad** entre variables independientes (X)
- ✅ Elimina **redundancia** antes de entrenar modelos
- ✅ Más **eficiente computacionalmente** (no requiere entrenar modelos)
- ✅ Evita **problemas** en modelos lineales sensibles a correlación
- ✅ Reduce **dimensionalidad** desde el inicio

## 📊 CÓDIGO: Análisis de Correlaciones

In [ ]:
# 1. CALCULAR MATRIZ DE CORRELACIÓN (solo con X_train_imputado)
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

corr_matrix = X_train_imputado.corr()

# 2. VISUALIZAR CON HEATMAP
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, linewidths=0.5, cbar_kws={"shrink": .8})
plt.title('Matriz de Correlación - Variables Independientes', fontsize=16, pad=15)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# 3. IDENTIFICAR PARES DE VARIABLES ALTAMENTE CORRELACIONADAS (|r| > 0.8)
umbral_correlacion = 0.8

# Crear lista para almacenar pares correlacionados
pares_correlacionados = []

for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        correlacion = corr_matrix.iloc[i, j]
        if abs(correlacion) > umbral_correlacion:
            pares_correlacionados.append({
                'Variable_1': corr_matrix.columns[i],
                'Variable_2': corr_matrix.columns[j],
                'Correlación': round(correlacion, 3)
            })

# Mostrar resultados
if len(pares_correlacionados) > 0:
    df_pares = pd.DataFrame(pares_correlacionados)
    print("="*70)
    print(f"⚠️ VARIABLES ALTAMENTE CORRELACIONADAS (|r| > {umbral_correlacion})")
    print("="*70)
    print(df_pares)
    print("\n💡 RECOMENDACIÓN: Eliminar una de las variables de cada par")
else:
    print("="*70)
    print(f"✅ No hay variables con correlación > {umbral_correlacion}")
    print("="*70)

In [ ]:
# 4. ELIMINAR VARIABLES CORRELACIONADAS
# Ejemplo: Si 'emp.var.rate' y 'euribor3m' están correlacionadas
# Decidir cuál eliminar basándote en interpretación o importancia previa

variables_a_eliminar = ['emp.var.rate', 'euribor3m']  # EJEMPLO - ajustar según tus datos

# Eliminar en train
X_train_limpio = X_train_imputado.drop(variables_a_eliminar, axis=1, errors='ignore')

# Eliminar en test
X_test_limpio = X_test_imputado.drop(variables_a_eliminar, axis=1, errors='ignore')

print(f"✅ Variables eliminadas: {variables_a_eliminar}")
print(f"📊 Forma original: {X_train_imputado.shape}")
print(f"📊 Forma nueva: {X_train_limpio.shape}")

---

# 🟡 PASO 5: VIF (Variance Inflation Factor)

## ¿Qué es VIF?
- Mide cuánto se "infla" la varianza de un coeficiente debido a la multicolinealidad
- **VIF = 1**: No hay correlación
- **VIF = 1-5**: Correlación moderada (aceptable)
- **VIF = 5-10**: Correlación alta (precaución)
- **VIF > 10**: Multicolinealidad severa (eliminar variable)

## 📊 CÓDIGO: Cálculo de VIF

In [ ]:
from statsmodels.tools.tools import add_constant
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Agregar constante (requerido para VIF)
X_vif = add_constant(X_train_limpio)

# Calcular VIF para cada variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) 
                   for i in range(X_vif.shape[1])]

# Eliminar la constante de los resultados
vif_data = vif_data[vif_data["Variable"] != "const"]

# Ordenar por VIF descendente
vif_data = vif_data.sort_values("VIF", ascending=False)

print("="*60)
print("📊 VARIANCE INFLATION FACTOR (VIF)")
print("="*60)
print(vif_data)
print("\n" + "="*60)

# Identificar variables problemáticas
variables_altas = vif_data[vif_data["VIF"] > 10]
if len(variables_altas) > 0:
    print(f"⚠️ VARIABLES CON VIF > 10 (Eliminar):")
    print(variables_altas)
else:
    print("✅ No hay variables con VIF > 10")

---

# 🔵 PASO 6: MÉTODOS DE SELECCIÓN DE VARIABLES (DESPUÉS)

## ¿Por qué después?
- ✅ Trabaja con variables **ya "limpias"** sin correlación
- ✅ Resultados **más confiables** (no sesgados por multicolinealidad)
- ✅ **Más rápido** (menos variables para analizar)
- ✅ Evita que el modelo "elija arbitrariamente" entre variables correlacionadas

## Métodos a usar:
1. **Random Forest Feature Importance**
2. **Permutation Importance**
3. **SHAP Values**

## 📊 MÉTODO 1: Random Forest Feature Importance

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Entrenar Random Forest SOLO con datos de train (sin correlaciones)
model_rf = RandomForestClassifier(random_state=42, n_jobs=-1).fit(X_train_limpio, Y_train)

# Obtener importancias
importances = model_rf.feature_importances_
importances_pct = (importances / importances.sum()) * 100

# Crear DataFrame
df_rf_imp = pd.DataFrame({
    'feature': X_train_limpio.columns,
    'rf_importance': importances_pct
}).sort_values(by='rf_importance', ascending=False)

# Calcular importancia acumulada
df_rf_imp['rf_importance_acum'] = df_rf_imp['rf_importance'].cumsum()

print("="*70)
print("🌲 RANDOM FOREST - FEATURE IMPORTANCE")
print("="*70)
print(df_rf_imp)
print("\n" + "="*70)

## 📊 MÉTODO 2: Permutation Importance

In [ ]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.inspection import permutation_importance

# Crear conjunto de validación
X_train1, X_val, y_train1, y_val = train_test_split(
    X_train_limpio, Y_train, test_size=0.2, random_state=42
)

# Entrenar XGBoost
model_xgb = XGBClassifier(objective='binary:logistic', random_state=42).fit(X_train1, y_train1)

# Calcular Permutation Importance
perm = permutation_importance(model_xgb, X_val, y_val, 
                              n_repeats=10, random_state=42, 
                              n_jobs=-1, scoring='roc_auc')

# Crear DataFrame
df_perm_imp = pd.DataFrame({
    'feature': X_train_limpio.columns,
    'perm_imp': perm.importances_mean * 100
}).sort_values('perm_imp', ascending=False)

print("="*70)
print("🔀 PERMUTATION IMPORTANCE")
print("="*70)
print(df_perm_imp)
print("\n" + "="*70)

## 📊 MÉTODO 3: SHAP Values

In [ ]:
import lightgbm as lgb
import shap

# Entrenar LightGBM
model_lgbm = lgb.LGBMClassifier(random_state=42, n_jobs=-1).fit(X_train1, y_train1)

# Calcular SHAP values
explainer = shap.Explainer(model_lgbm, X_val)
shap_vals = explainer(X_val).values

# Importancia promedio absoluta
imp_shap = np.abs(shap_vals).mean(axis=0)
imp_shap_pct = (imp_shap / imp_shap.sum()) * 100

# Crear DataFrame
df_shap_imp = pd.DataFrame({
    "feature": X_val.columns,
    "shap_imp": imp_shap_pct
}).sort_values('shap_imp', ascending=False)

print("="*70)
print("🎯 SHAP VALUES - FEATURE IMPORTANCE")
print("="*70)
print(df_shap_imp)
print("\n" + "="*70)

# Gráfico SHAP
shap.summary_plot(shap_vals, X_val, plot_type="bar")

## 🔗 COMBINACIÓN DE LOS 3 MÉTODOS

In [ ]:
# Combinar los 3 métodos en un solo DataFrame
df_importances = (
    df_rf_imp
    .merge(df_perm_imp, on='feature', how='outer')
    .merge(df_shap_imp, on='feature', how='outer')
).sort_values('rf_importance', ascending=False)

# Calcular promedio de importancia
df_importances['importancia_promedio'] = df_importances[
    ['rf_importance', 'perm_imp', 'shap_imp']
].mean(axis=1)

# Ordenar por promedio
df_importances_sorted = df_importances.sort_values('importancia_promedio', ascending=False)

print("="*90)
print("📊 COMBINACIÓN DE LOS 3 MÉTODOS DE SELECCIÓN")
print("="*90)
print(df_importances_sorted.round(2))
print("\n" + "="*90)

# Filtrar variables con importancia promedio >= 2%
umbral_importancia = 2.0
df_filtradas = df_importances_sorted[df_importances_sorted['importancia_promedio'] >= umbral_importancia]

print(f"\n✅ VARIABLES SELECCIONADAS (Importancia promedio >= {umbral_importancia}%):")
print("="*90)
print(df_filtradas[['feature', 'importancia_promedio']].round(2))
print(f"\n📊 Total de variables seleccionadas: {len(df_filtradas)} de {len(df_importances_sorted)}")

---

# ⚠️ RESUMEN: ¿QUÉ PASA SI LO HACES AL REVÉS?

## ❌ PROBLEMA: Selección → Correlaciones

### Consecuencias:
1. **Resultados inconsistentes**: El modelo puede elegir arbitrariamente entre variables correlacionadas
2. **Mayor tiempo de cómputo**: Entrenas con variables redundantes
3. **Interpretación difícil**: Dos variables dicen lo mismo
4. **Sesgo en importancia**: Variables correlacionadas compiten entre sí

### Ejemplo:
```
Si Glucose y Insulin_log están correlacionadas (r = 0.85):

❌ MAL (Selección primero):
  - Random Forest elige Glucose como importante (85%)
  - En otra ejecución elige Insulin_log (82%)
  - Resultados inestables

✅ BIEN (Correlaciones primero):
  - Eliminas Insulin_log
  - Random Forest solo ve Glucose
  - Resultados estables y confiables
```

---

# 📋 CHECKLIST FINAL

## ✅ Orden Correcto:

| Paso | Acción | Herramienta | ¿Cuándo? |
|------|--------|------------|----------|
| 1 | Limpieza básica | Pandas | Antes del split |
| 2 | **Train/Test Split** | `train_test_split` | **AQUÍ** |
| 3 | Imputación | MissForest/SimpleImputer | Solo fit en train |
| 4 | **Correlaciones** | `corr()`, heatmap | **PRIMERO** (solo train) |
| 5 | VIF | `variance_inflation_factor` | Después correlaciones |
| 6 | **Selección Variables** | RF, Permutation, SHAP | **DESPUÉS** (solo train) |
| 7 | Escalado | StandardScaler | Fit en train, transform en test |
| 8 | Modelado | LogisticRegression, etc. | Con variables seleccionadas |

---

## 🎯 REGLA DE ORO:

**TODO lo que "aprende" de los datos debe hacerse SOLO con train después del split**

- ✅ **Correlaciones**: Calculadas con `X_train_imputado`
- ✅ **VIF**: Calculado con `X_train_limpio`
- ✅ **Feature Importance**: Modelos entrenados con `X_train_limpio`
- ✅ **Escalado**: `scaler.fit()` solo con `X_train`

---

## 💡 TIP FINAL:

En tu proyecto de diabetes o clasificación bancaria:

1. Divide primero con `train_test_split`
2. Imputa valores faltantes (fit solo en train)
3. Analiza correlaciones y elimina variables redundantes
4. Calcula VIF para confirmar
5. Aplica RF + Permutation + SHAP para seleccionar las mejores
6. Escala y modela 🚀

---

# 🔄 TRANSFORMACIONES: ¿ANTES O DESPUÉS DEL SPLIT?

## ✅ TRANSFORMACIONES ACEPTABLES ANTES DEL SPLIT

### Regla: Transformaciones que NO aprenden de los datos

Estas son **determinísticas** y se pueden aplicar antes del split sin riesgo de data leakage:

In [ ]:
# ✅ EJEMPLOS DE TRANSFORMACIONES ANTES DEL SPLIT:

# 1. TRANSFORMACIONES MATEMÁTICAS DETERMINÍSTICAS
df['Insulin_log'] = np.log1p(df['Insulin'])  # Logaritmo
df['BMI_sqrt'] = np.sqrt(df['BMI'])  # Raíz cuadrada
df['Age_squared'] = df['Age'] ** 2  # Potencia
df['DiabetesPedigreeFunction_sqrt'] = np.sqrt(df['DiabetesPedigreeFunction'])

# 2. CODIFICACIONES PREDEFINIDAS (MANUAL)
# Mapping ordinal manual
df['month'] = df['month'].map({
    'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 
    'may': 5, 'jun': 6, 'jul': 7, 'aug': 8,
    'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
})

df['education'] = df['education'].map({
    'illiterate': 1,
    'basic.4y': 2,
    'basic.6y': 2,
    'basic.9y': 2,
    'high.school': 3,
    'professional.course': 4,
    'university.degree': 4
})

# 3. REEMPLAZAR VALORES IMPOSIBLES POR NaN
df['BloodPressure'] = df['BloodPressure'].replace(0, np.nan)
df['Glucose'] = df['Glucose'].replace(0, np.nan)
df['SkinThickness'] = df['SkinThickness'].replace(0, np.nan)

# 4. CREAR VARIABLES DERIVADAS
df['BMI_category'] = pd.cut(df['BMI'], bins=[0, 18.5, 25, 30, 100], 
                             labels=[1, 2, 3, 4])

print("✅ Transformaciones determinísticas aplicadas antes del split")
print("✅ No hay data leakage porque no se aprende de los datos")

### ¿Por qué es seguro?

- ✅ **No hay data leakage** porque no usas estadísticas de los datos (media, mediana, moda, etc.)
- ✅ La transformación es **igual para todos los datos** (no depende del conjunto)
- ✅ Es **reproducible** y **determinística** (siempre da el mismo resultado)
- ✅ No contamina el conjunto de test con información del conjunto de train

---

## ❌ TRANSFORMACIONES QUE DEBEN IR DESPUÉS DEL SPLIT

### Regla: Transformaciones que APRENDEN estadísticas de los datos

Estas deben hacerse solo con train y luego aplicarse a test:

In [ ]:
# ❌ EJEMPLOS DE TRANSFORMACIONES DESPUÉS DEL SPLIT:

# 1. IMPUTACIÓN CON ESTADÍSTICAS
from sklearn.impute import SimpleImputer

# ❌ MAL - Data leakage
imputer = SimpleImputer(strategy='median')
df_imputed = imputer.fit_transform(df)  # ← Aprende mediana de TODO el dataset
X_train, X_test = train_test_split(df_imputed)  # ← Test contaminado

# ✅ BIEN - Sin data leakage
X_train, X_test = train_test_split(df)
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)  # ← Aprende solo de train
X_test_imp = imputer.transform(X_test)  # ← Aplica lo aprendido de train


# 2. ESCALADO (STANDARDSCALER, MINMAXSCALER, ROBUSTSCALER)
from sklearn.preprocessing import StandardScaler

# ❌ MAL
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df)  # ← Aprende media/std de TODO
X_train, X_test = train_test_split(df_scaled)

# ✅ BIEN
X_train, X_test = train_test_split(df)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)  # ← Aprende de train
X_test_sc = scaler.transform(X_test)  # ← Usa parámetros de train


# 3. TARGET ENCODING
# ❌ MAL - Aprende tasas de conversión del dataset completo
target_encoding = df.groupby('job')['y'].mean()
df['job_encoded'] = df['job'].map(target_encoding)
X_train, X_test = train_test_split(df)

# ✅ BIEN - Aprende solo de train
X_train, X_test = train_test_split(df)
target_encoding = X_train.groupby('job')['y'].mean()
X_train['job_encoded'] = X_train['job'].map(target_encoding)
X_test['job_encoded'] = X_test['job'].map(target_encoding)


print("❌ Estas transformaciones causan DATA LEAKAGE si se hacen antes del split")
print("✅ Siempre hacer fit() solo con train, luego transform() en test")

---

## ⚠️ CASO ESPECIAL: Reemplazar 0 por Mediana

### Situación: Valores imposibles (ej: Glucosa = 0)

**Técnicamente mejor después del split, pero impacto mínimo si se hace antes:**

In [ ]:
# ⚠️ OPCIÓN 1: ANTES DEL SPLIT (Impacto mínimo pero técnicamente no ideal)
mediana_bp = df['BloodPressure'].median()  # ← Usa info de train Y test
df['BloodPressure'] = df['BloodPressure'].replace(0, mediana_bp)
X_train, X_test = train_test_split(df)

print(f"⚠️ Mediana calculada con todo el dataset: {mediana_bp}")
print("Impacto: Mínimo, pero técnicamente hay mini data leakage")


# ✅ OPCIÓN 2: DESPUÉS DEL SPLIT (Ideal, sin data leakage)
X_train, X_test = train_test_split(df)

# Calcular mediana solo con train
mediana_bp_train = X_train['BloodPressure'][X_train['BloodPressure'] > 0].median()

# Aplicar a ambos conjuntos
X_train['BloodPressure'] = X_train['BloodPressure'].replace(0, mediana_bp_train)
X_test['BloodPressure'] = X_test['BloodPressure'].replace(0, mediana_bp_train)

print(f"\n✅ Mediana calculada solo con train: {mediana_bp_train}")
print("Sin data leakage - Test no contamina train")


# ALTERNATIVA: Reemplazar por NaN antes del split (mejor práctica)
df['BloodPressure'] = df['BloodPressure'].replace(0, np.nan)
df['Glucose'] = df['Glucose'].replace(0, np.nan)
# Luego dividir y usar imputer (que aprenderá solo de train)

---

## 📊 TABLA RESUMEN

| Transformación | ¿Antes del split? | Razón |
|----------------|-------------------|-------|
| **`np.log1p()`** | ✅ **Sí** | No aprende de los datos - Determinística |
| **`np.sqrt()`** | ✅ **Sí** | No aprende de los datos - Determinística |
| **`x ** 2`** | ✅ **Sí** | No aprende de los datos - Determinística |
| **Mapeo manual** (1,2,3) | ✅ **Sí** | No aprende de los datos - Predefinido |
| **One-Hot Encoding** | ✅ **Sí** | Si las categorías son conocidas |
| **Reemplazar por NaN** | ✅ **Sí** | No aprende estadísticas |
| **Target Encoding** | ⚠️ **Después** | Aprende tasas del target (data leakage) |
| **Reemplazar 0 por mediana** | ⚠️ **Después (ideal)** | Aprende estadística del dataset |
| **StandardScaler** | ❌ **Después** | Aprende media y desviación estándar |
| **MinMaxScaler** | ❌ **Después** | Aprende mínimo y máximo |
| **RobustScaler** | ❌ **Después** | Aprende cuartiles |
| **SimpleImputer** | ❌ **Después** | Aprende mediana/media/moda |
| **MissForest** | ❌ **Después** | Aprende patrones de imputación |
| **KNNImputer** | ❌ **Después** | Aprende vecinos cercanos |

---

## 🎯 REGLA DE ORO SIMPLIFICADA

### ¿Usa `.fit()` o calcula estadísticas (media, mediana, moda, etc.)?

- **NO** → ✅ Puedes hacerlo **ANTES** del split
- **SÍ** → ❌ Debes hacerlo **DESPUÉS** del split (solo fit con train)

---

## 💡 EJEMPLO PRÁCTICO: Proyecto de Diabetes

```python
# ✅ ANTES DEL SPLIT
df['Insulin_log'] = np.log1p(df['Insulin'])  # Transformación matemática
df['BMI_sqrt'] = np.sqrt(df['BMI'])  # Transformación matemática
df['BloodPressure'] = df['BloodPressure'].replace(0, np.nan)  # Marcar como faltante

# ⭐ DIVIDIR
X = df.drop('Outcome', axis=1)
Y = df['Outcome']
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# ❌ DESPUÉS DEL SPLIT
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)  # fit solo con train
X_test_imp = imputer.transform(X_test)  # transform con parámetros de train

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_imp)  # fit solo con train
X_test_sc = scaler.transform(X_test_imp)  # transform con parámetros de train
```